In [25]:
## Dependencies

import pandas as pd
import requests
from bs4 import BeautifulSoup
import trafilatura
from urllib.parse import urljoin, urlparse, parse_qs
import os

## Step 1: Getting the data


In [26]:
df = pd.read_csv('data/AI-use-LibGuides.csv')
libguides = df[df['Type'] == 'LibGuide']

In [27]:
libguides

,Institution,LibGuideURL,Type
0,USC,https://libguides.usc.edu/c.php?g=1457247&p=10...,LibGuide
1,William & Mary,https://guides.libraries.wm.edu/c.php?g=131720...,LibGuide
2,UArkansas,https://libguides.uark.edu/AI/databases,LibGuide
3,Auburn University,https://libguides.auburn.edu/c.php?g=1444337&p...,LibGuide
5,Baylor University,https://libguides.baylor.edu/c.php?g=1441620&p...,LibGuide
6,UCIrvine,https://guides.lib.uci.edu/gen-ai,LibGuide
7,UCSD,https://ucsd.libguides.com/AI,LibGuide
8,UCMerced,https://libguides.ucmerced.edu/artificial-inte...,LibGuide
10,UCDavis,https://guides.library.ucdavis.edu/genai,LibGuide
11,UCSB,https://guides.library.ucsb.edu/ai,LibGuide


## Step 2: Get guide links

In [28]:
def get_guide_links(base_url, html):
    """Find all tab links that belong to the same LibGuide."""
    soup = BeautifulSoup(html, 'html.parser')
    
    # Parse the starting URL to understand its structure
    parsed_base = urlparse(base_url)
    domain = parsed_base.netloc
    base_path = parsed_base.path
    
    # If it's a c.php link, extract the Guide ID ('g' parameter)
    base_g = parse_qs(parsed_base.query).get('g', [None])[0]
    
    links = set([base_url]) # Always include the starting page
    for a in soup.find_all('a', href=True):
        href = a['href']
        full_url = urljoin(base_url, href)  # type: ignore
        parsed_full = urlparse(full_url)
        
        # Must be the same domain
        if parsed_full.netloc != domain:
            continue
            
        # Avoid print-mode pages or admin login links
        if 'print' in full_url.lower() or 'admin' in full_url.lower():
            continue
            
        # Strategy A: c.php guides (match the 'g' parameter)
        if base_path == '/c.php' and parsed_full.path == '/c.php':
            full_g = parse_qs(parsed_full.query).get('g', [None])[0]
            if full_g == base_g and full_g is not None:
                links.add(full_url.split('#')[0]) # Remove anchor tags
                
        # Strategy B: Friendly URLs (e.g., /genai matching /genai/cases)
        elif base_path != '/' and base_path != '/c.php':
            # Case-insensitive check to ensure the sub-page belongs to the base path
            if parsed_full.path.lower().startswith(base_path.lower()):
                links.add(full_url.split('#')[0]) # Remove anchor tags
                
    return list(links)

## Step 3: Scraping function

In [29]:
def scrape_libguide(start_url, dry_run=False) -> str | None:
    """Scrape the main page and all its tabs, extracting only main text."""
    try:
        response = requests.get(start_url, timeout=10)
        response.raise_for_status()
    except requests.RequestException:
        return ""

    # Get all sub-pages for this guide
    tab_urls = get_guide_links(start_url, response.text)
    
    guide_text = []
    for url in tab_urls:
        try:
            res = requests.get(url, timeout=10)
            if dry_run:
                print(url)
                continue
            # Trafilatura magically finds the main content and ignores the sidebar/nav!
            extracted_text = trafilatura.extract(res.text)
            if extracted_text:
                guide_text.append(extracted_text)
        except requests.RequestException:
            continue
            
    # Combine all tabs into one document separated by newlines
    return "\n\n".join(guide_text)

## Step 4: Scrape!

In [30]:
# Convert filtered dataframe to a Dict object
guides_list = libguides.to_dict(orient="records")

for guide in guides_list:
    inst = guide.get("Institution")
    outputfile = f"{inst.split()[0]}.txt"
    dest = os.path.join("data", outputfile)
    # prevent unnecessary requests
    if os.path.exists(dest):
        continue
    print(inst)
    scraped_text = scrape_libguide(guide.get("LibGuideURL"))
    
    with open(dest, "w") as f:
        f.write(scraped_text)

print("Done!")        

Done!
